# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [28]:
# Write your code below.
from dotenv import load_dotenv
import os
import shutil

# Now loading our. env file.
load_dotenv()

# Creating a variable that stores PRICE_DATA environment.
price_data = os.getenv("PRICE_DATA")

# We need to double check if PRICE_DATA has data.
if os.path.exists(price_data):
    print("PRICE_DATA loaded successfully.")
    print(price_data)
else:
    print("PRICE_DATA not found. Ensure you have executed the prerequisite notebook.")


PRICE_DATA loaded successfully.
../../05_src/data/prices/


In [29]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [30]:
import os
from glob import glob

# Write your code below.

parquet_files = glob(os.path.join(price_data, "**/*.parquet"), recursive = True)
#print(parquet_files)

# Print out the first parquet file's dataset.
ddf = dd.read_parquet(parquet_files[0])
ddf.compute()


Price,Date,Adj Close,Close,High,Low,Open,Volume,Year
Ticker,,,,,,,,
CTAS,2002-01-02,NaN,5.070603,5.109207,4.913060,5.055997,3887200.0,2002
CTAS,2002-01-03,NaN,5.143637,5.148854,5.005917,5.110251,2285600.0,2002
CTAS,2002-01-04,NaN,5.145725,5.173895,5.011135,5.135292,1502400.0,2002
CTAS,2002-01-07,NaN,5.106078,5.216672,5.100862,5.136335,2756800.0,2002
CTAS,2002-01-08,NaN,5.211456,5.232323,5.117556,5.138422,2806400.0,2002
...,...,...,...,...,...,...,...,...
CTAS,2002-12-24,NaN,4.978729,5.002168,4.954224,4.995776,1637200.0,2002
CTAS,2002-12-26,NaN,4.974468,5.133216,4.956356,4.976599,2782000.0,2002
CTAS,2002-12-27,NaN,4.862598,4.994711,4.839159,4.960618,3041600.0,2002


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.

# Reading the first 10 parquet files.
# Reason: It would take a very long time to get all the data, so I am using the first 10 files.
#ddf = dd.read_parquet(parquet_files[0:10]).set_index("Ticker")
ddf = dd.read_parquet(parquet_files).set_index("Ticker")

# Now we need to add the 'lag' feature for Close and Adj_Close according to the question description.
ddf_lags = ddf.groupby('Ticker', group_keys=False).apply(lambda x: x.assign(Close_lag_1 = x['Close'].shift(1), Close_lag_2 = x['Adj Close'].shift(1)))

# Now we calculate the range by using the high subtracting from the low.
ddf_feat = ddf_lags.groupby('Ticker', group_keys=False).apply(lambda x: x.assign(returns = (x['Close'] / x['Close_lag_1']) - 1, hi_lo_range = x['High'] - x['Low'])
)

dd_feat = ddf_feat

#dd_feat.compute()


/var/folders/t9/j9cv6kkj7pg15qwk9ljc1ttr0000gn/T/ipykernel_80606/1821988575.py:9: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  ddf_lags = ddf.groupby('Ticker', group_keys=False).apply(lambda x: x.assign(Close_lag_1 = x['Close'].shift(1), Close_lag_2 = x['Adj Close'].shift(1)))


NameError: name 'df_lags' is not defined

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [27]:
# Write your code below.

# Map reduce concept.
dd_feat = dd_feat.groupby("Ticker").apply(lambda x: x.assign(ma_return = x["returns"].rolling(10).mean()))

dd_feat.compute()


/var/folders/t9/j9cv6kkj7pg15qwk9ljc1ttr0000gn/T/ipykernel_80606/3928444153.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd_feat.groupby("Ticker").apply(lambda x: x.assign(ma_return = x["returns"].rolling(10).mean()))


Price                             Date  Adj Close      Close       High  \
Ticker Ticker Ticker Ticker                                               
CTAS   CTAS   CTAS   CTAS   2021-01-04        NaN  77.999481  80.528332   
                     CTAS   2021-01-05        NaN  79.200363  79.234413   
                     CTAS   2021-01-06        NaN  79.677071  80.950582   
                     CTAS   2021-01-07        NaN  80.600967  81.100385   
                     CTAS   2021-01-08        NaN  79.486389  81.370549   
...                                ...        ...        ...        ...   
                     CTAS   2002-12-24        NaN   4.978729   5.002168   
                     CTAS   2002-12-26        NaN   4.974468   5.133216   
                     CTAS   2002-12-27        NaN   4.862598   4.994711   
                     CTAS   2002-12-30        NaN   4.922262   4.943570   
                     CTAS   2002-12-31        NaN   4.874318   4.973402   

Price                              Low       Open     Volume  Year  \
Ticker Ticker Ticker Ticker                                          
CTAS   CTAS   CTAS   CTAS    77.704368  80.346729  3400400.0  2021   
                     CTAS    76.823603  77.681688  2030000.0  2021   
                     CTAS    79.279809  79.452336  1658800.0  2021   
                     CTAS    79.234390  79.874543  2804400.0  2021   
                     CTAS    79.204903  80.342209  1593600.0  2021   
...                                ...        ...        ...   ...   
                     CTAS     4.954224   4.995776  1637200.0  2002   
                     CTAS     4.956356   4.976599  2782000.0  2002   
                     CTAS     4.839159   4.960618  3041600.0  2002   
                     CTAS     4.805065   4.863663  4188000.0  2002   
                     CTAS     4.784822   4.906280  5323200.0  2002   

Price                        Close_lag_1  Adj_Close_lag_1   returns  \
Ticker Ticker Ticker Ticker                                           
CTAS   CTAS   CTAS   CTAS            NaN              NaN       NaN   
                     CTAS      37.263641              NaN  1.125406   
                     CTAS      36.667389              NaN  1.172968   
                     CTAS      38.391632              NaN  1.099441   
                     CTAS      38.456158              NaN  1.066935   
...                                  ...              ...       ...   
                     CTAS       4.811263              NaN -0.010519   
                     CTAS       4.760652              NaN  0.009013   
                     CTAS       4.803559              NaN  0.007330   
                     CTAS       4.838769              NaN -0.001365   
                     CTAS       4.832166              NaN -0.001366   

Price                        hi_lo_range  ma_return  
Ticker Ticker Ticker Ticker                          
CTAS   CTAS   CTAS   CTAS       2.823963        NaN  
                     CTAS       2.410810        NaN  
                     CTAS       1.670773        NaN  
                     CTAS       1.865995        NaN  
                     CTAS       2.165646        NaN  
...                                  ...        ...  
                     CTAS       0.125425  -0.004926  
                     CTAS       0.111122  -0.004047  
                     CTAS       0.088018  -0.005116  
                     CTAS       0.082517  -0.005187  
                     CTAS       0.082516  -0.004914  

[2519 rows x 13 columns]

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

Kevin's Answer: No, converting the Dask DataFrame to a Pandas DataFrame was not strictly required to compute the moving average. Since Dask supports rolling operations, we could have directly used .rolling(10).mean() within Dask.

+ Would it have been better to do it in Dask? Why?

Kevin's Answer: Yes, by computing the moving average in Dask, it would have been a better approach, particularly for large datasets. Dask offers greater scalability by handling data that exceeds memory limits, whereas Pandas requires loading everything into RAM. Additionally, Dask leverages parallelism by distributing computations across multiple cores or even a cluster, while Pandas operates in a single-threaded manner. This makes Dask more efficient, as converting a large dataset to Pandas can be both slow and memory-intensive.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.